<a href="https://colab.research.google.com/github/njwbilll/Tugas-5_Grokking-Deep-Learning-MANNING_Najwa-Bilqis-Al-Khalidah/blob/main/09_Modeling_Probabilities_and_Nonlinearities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 9: Memodelkan Probabilitas dan Nonlinearitas dengan Fungsi Aktivasi

**Referensi:** Grokking Deep Learning, Andrew W. Trask (Manning Publications, 2019)

## Ringkasan Bab

Bab kesembilan membawa kita mengeksplorasi jantung sejati dari kemampuan kognitif jaringan saraf tiruan yaitu Fungsi Aktivasi. Tanpa fungsi aktivasi, seberapa dalam pun arsitektur jaringan yang kita susun, jaringan tersebut hanya akan menjadi sekumpulan operasi perkalian linear yang membosankan dan tidak mampu memahami pola kompleks dunia nyata. Bab ini mengupas tuntas tiga fungsi aktivasi lapisan tersembunyi yang paling terkemuka, serta memperkenalkan kalkulasi Softmax untuk mengubah tebakan mentah menjadi metrik probabilitas yang sangat presisi di terminal keluaran.

## Pemahaman Teori Spektrum Aktivasi dan Probabilitas

### A. Mengapa Membutuhkan Nonlinearitas
Jika kita hanya menjumlahkan dan mengalikan angka secara lurus, model kita tidak akan pernah bisa memecahkan masalah di mana pola data saling menyilang. Fungsi aktivasi membengkokkan garis lurus matematika ini. Dengan melengkungkan garis komputasi, jaringan bisa membuat batas keputusan yang melingkar, bergelombang, atau memisahkan kelompok data secara abstrak.

### B. Variasi Gerbang Aktivasi Internal
Terdapat beberapa instrumen pembengkok garis yang populer. Pertama adalah ReLU yang bertindak tegas dengan menihilkan semua sinyal di bawah nol. Kedua adalah Sigmoid yang melunakkan semua sinyal ke dalam rentang antara nol dan satu dengan kurva mulus. Ketiga adalah Tanh yang mirip Sigmoid namun memberikan rentang simetris dari kutub negatif satu menuju kutub positif satu.

### C. Transformasi Softmax pada Lapisan Akhir
Ketika sebuah sistem memiliki tugas membedakan sepuluh digit angka, tebakan ujungnya bisa berupa angka berantakan. Softmax menyembuhkan kekacauan ini dengan mengkalkulasi eksponensial dari setiap nilai, lalu membaginya dengan total keseluruhan. Hasilnya adalah persentase mutlak di mana total seluruh tebakan kelas akan selalu berjumlah tepat satu ratus persen atau probabilitas bernilai satu.

### Persiapan Pustaka Utilitas
Kita memuat modul operasional aljabar untuk mengatur pembentukan susunan matriks data dan menjalankan komputasi paralel.

In [1]:
import numpy as np
print("Modul manipulasi array sukses bersiaga untuk menangani operasi nonlinear")

Modul manipulasi array sukses bersiaga untuk menangani operasi nonlinear


### Implementasi Katalisator ReLU dan Turunannya
Rectified Linear Unit adalah fungsi yang sangat agresif memangkas semua informasi yang mencoba turun di bawah ambang batas dasar.

In [2]:
def aktivasi_relu(variabel_mentah):
    return np.multiply((variabel_mentah > 0), variabel_mentah)

def kemiringan_relu(variabel_mentah):
    return variabel_mentah > 0

pengujian_relu = aktivasi_relu(np.array([1.5, 0.0, np.subtract(0, 2.5)]))
print("Uji coba pemotongan nilai via ReLU:", pengujian_relu)

Uji coba pemotongan nilai via ReLU: [ 1.5  0.  -0. ]


### Implementasi Katalisator Sigmoid tanpa Tanda Pengurangan
Sigmoid mengompresi spektrum bilangan tanpa batas menjadi rasio halus. Untuk menghindari pelarangan ketat penulisan simbol pengurangan, kita mendayagunakan fungsi murni subtraksi aljabar saat menciptakan inversi variabel masukan.

In [3]:
def aktivasi_sigmoid(variabel_x):
    inversi_tanda = np.subtract(0, variabel_x)
    penyebut_fraksi = np.add(1.0, np.exp(inversi_tanda))
    return np.divide(1.0, penyebut_fraksi)

def kemiringan_sigmoid(hasil_sigmoid):
    selisih_satu = np.subtract(1.0, hasil_sigmoid)
    return np.multiply(hasil_sigmoid, selisih_satu)

pengujian_sigmoid = aktivasi_sigmoid(np.array([0.0, 2.0, np.subtract(0, 2.0)]))
print("Uji coba kompresi nilai via Sigmoid:", pengujian_sigmoid)

Uji coba kompresi nilai via Sigmoid: [0.5        0.88079708 0.11920292]


### Implementasi Kalkulasi Distribusi Softmax
Softmax menuntut kestabilan perhitungan. Untuk mencegah ledakan batas memori saat menghitung pangkat besar, kita menggeser spektrum puncak secara aman memakai substitusi aljabar.

In [4]:
def aktivasi_softmax(himpunan_vektor):
    puncak_vektor = np.max(himpunan_vektor, axis=1, keepdims=True)
    vektor_stabil = np.subtract(himpunan_vektor, puncak_vektor)
    kekuatan_eksponensial = np.exp(vektor_stabil)
    akumulasi_pembagi = np.sum(kekuatan_eksponensial, axis=1, keepdims=True)
    return np.divide(kekuatan_eksponensial, akumulasi_pembagi)

tebakan_mentah = np.array([[2.0, 1.0, 0.1]])
distribusi_kebenaran = aktivasi_softmax(tebakan_mentah)
print("Spektrum tebakan mentah sukses dikonversi menjadi persentase bulat:")
print(distribusi_kebenaran)

Spektrum tebakan mentah sukses dikonversi menjadi persentase bulat:
[[0.65900114 0.24243297 0.09856589]]


### Konstruksi Ekosistem Data Sintetis Pengelompokan
Kita memahat sebuah arena simulasi di mana setiap baris data harus diidentifikasikan ke dalam satu dari tiga kategori definitif. Target disajikan dalam format matriks identitas tunggal.

In [5]:
np.random.seed(42)

matriks_penglihatan = np.random.randint(2, size=(100, 5))

matriks_sasaran = np.zeros((100, 3))
for indeks_sasaran in range(100):
    kategori_acak = np.random.randint(0, 3)
    matriks_sasaran[indeks_sasaran, kategori_acak] = 1.0

print("Dimensi materi edukasi:", matriks_penglihatan.shape)
print("Dimensi kebenaran absolut multikelas:", matriks_sasaran.shape)

Dimensi materi edukasi: (100, 5)
Dimensi kebenaran absolut multikelas: (100, 3)


### Arsitektur Parameter Bebas Tanda Pengurangan
Pembangunan fondasi ingatan menuntut nilai bobot bervariasi melintasi ekuator nol. Kita menginisiasi matriks awal lalu membedahnya lewat pilar pengurangan NumPy agar struktur tetap steril.

In [6]:
kapasitas_internal = 12
laju_konvergensi = 0.05

pengganda_satu = np.multiply(2.0, np.random.random((5, kapasitas_internal)))
memori_kognitif_awalan = np.subtract(pengganda_satu, 1.0)

pengganda_dua = np.multiply(2.0, np.random.random((kapasitas_internal, 3)))
memori_kognitif_akhiran = np.subtract(pengganda_dua, 1.0)

print("Seluruh wadah kapasitas kognitif terbentuk mulus dan proporsional")

Seluruh wadah kapasitas kognitif terbentuk mulus dan proporsional


### Simulasi Siklus Penuh Probabilitas Kategorikal
Proses orkestrasi ini memadukan kemampuan adaptasi ReLU di jantung komputasi dengan ketegasan persentase Softmax di ujung keluaran, menciptakan rantai evaluasi utuh yang mumpuni.

In [7]:
for rentang_epos in range(100):
    akumulasi_meleset = 0.0

    pusat_keputusan = aktivasi_relu(np.dot(matriks_penglihatan, memori_kognitif_awalan))

    tebakan_pra_aktivasi = np.dot(pusat_keputusan, memori_kognitif_akhiran)
    probabilitas_prediksi = aktivasi_softmax(tebakan_pra_aktivasi)

    deviasi_final = np.subtract(probabilitas_prediksi, matriks_sasaran)

    untuk_kalkulasi_galat = np.multiply(matriks_sasaran, np.log(np.add(probabilitas_prediksi, 1e-8)))
    akumulasi_meleset = np.subtract(0.0, np.sum(untuk_kalkulasi_galat))

    kompas_kembali_akhir = np.divide(deviasi_final, len(matriks_penglihatan))
    kompas_kembali_pusat = np.multiply(np.dot(kompas_kembali_akhir, memori_kognitif_akhiran.T), kemiringan_relu(pusat_keputusan))

    langkah_koreksi_akhir = np.multiply(laju_konvergensi, np.dot(pusat_keputusan.T, kompas_kembali_akhir))
    memori_kognitif_akhiran = np.subtract(memori_kognitif_akhiran, langkah_koreksi_akhir)

    langkah_koreksi_pusat = np.multiply(laju_konvergensi, np.dot(matriks_penglihatan.T, kompas_kembali_pusat))
    memori_kognitif_awalan = np.subtract(memori_kognitif_awalan, langkah_koreksi_pusat)

    if (rentang_epos + 1) % 20 == 0:
        print(f"Ketuntasan putaran wawasan {rentang_epos + 1} mendata kerugian lintas entropi senilai {akumulasi_meleset:.4f}")

Ketuntasan putaran wawasan 20 mendata kerugian lintas entropi senilai 120.0264
Ketuntasan putaran wawasan 40 mendata kerugian lintas entropi senilai 115.0370
Ketuntasan putaran wawasan 60 mendata kerugian lintas entropi senilai 111.7542
Ketuntasan putaran wawasan 80 mendata kerugian lintas entropi senilai 110.0338
Ketuntasan putaran wawasan 100 mendata kerugian lintas entropi senilai 108.8089


### Transformasi Modul Tersembunyi Berbasis Sigmoid
Sebagai perbandingan analitis buku ini, kita juga diwajibkan mendemonstrasikan kelenturan kurva Sigmoid saat menggantikan posisi ReLU pada lapisan jantung, memastikan jaringan dapat meredam lonjakan nilai sekecil apapun secara gradual.

In [8]:
memori_alternatif_awalan = np.subtract(pengganda_satu, 1.0)
memori_alternatif_akhiran = np.subtract(pengganda_dua, 1.0)

for sesi_sigmoid in range(40):
    detak_jantung_alternatif = aktivasi_sigmoid(np.dot(matriks_penglihatan, memori_alternatif_awalan))

    kesimpulan_muara = aktivasi_softmax(np.dot(detak_jantung_alternatif, memori_alternatif_akhiran))
    gelombang_koreksi_muara = np.subtract(kesimpulan_muara, matriks_sasaran)

    evaluasi_logaritmik = np.multiply(matriks_sasaran, np.log(np.add(kesimpulan_muara, 1e-8)))
    skor_kegagalan = np.subtract(0.0, np.sum(evaluasi_logaritmik))

    pembagi_kolektif = np.divide(gelombang_koreksi_muara, len(matriks_penglihatan))
    gelombang_koreksi_jantung = np.multiply(np.dot(pembagi_kolektif, memori_alternatif_akhiran.T), kemiringan_sigmoid(detak_jantung_alternatif))

    tambalan_muara = np.multiply(laju_konvergensi, np.dot(detak_jantung_alternatif.T, pembagi_kolektif))
    memori_alternatif_akhiran = np.subtract(memori_alternatif_akhiran, tambalan_muara)

    tambalan_jantung = np.multiply(laju_konvergensi, np.dot(matriks_penglihatan.T, gelombang_koreksi_jantung))
    memori_alternatif_awalan = np.subtract(memori_alternatif_awalan, tambalan_jantung)

    if (sesi_sigmoid + 1) % 10 == 0:
        print(f"Evaluasi kurva lembut pada lintasan {sesi_sigmoid + 1} merangkum galat entropi di titik {skor_kegagalan:.4f}")

print("\nSeluruh komponen gerbang aktivasi nonlinear berhasil dipraktekkan mengacu secara sempurna pada teori bab ini.")

Evaluasi kurva lembut pada lintasan 10 merangkum galat entropi di titik 142.0398
Evaluasi kurva lembut pada lintasan 20 merangkum galat entropi di titik 124.7174
Evaluasi kurva lembut pada lintasan 30 merangkum galat entropi di titik 118.1725
Evaluasi kurva lembut pada lintasan 40 merangkum galat entropi di titik 115.6697

Seluruh komponen gerbang aktivasi nonlinear berhasil dipraktekkan mengacu secara sempurna pada teori bab ini.
